In [1]:
import numpy as np

In [2]:
def euclidean_distance(x1, x2):
    """
    Calculate the Euclidean distance between two points.
    """
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)

    return np.sqrt(np.sum((x1 - x2) ** 2))

In [3]:
def manhattan_distance(x1, x2):
    """
    Calculate the Manhattan distance between two points.
    """
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)

    return np.sum(np.abs(x1 - x2))

In [4]:
A = np.array([2, 3])
B = np.array([7, 9])

print("Euclidean:", euclidean_distance(A, B))
print("Manhattan:", manhattan_distance(A, B))

Euclidean: 7.810249675906654
Manhattan: 11


In [5]:
X_train = np.array([
    [1, 1],
    [1, 2],
    [2, 1],
    [5, 5],
    [6, 5],
    [5, 6]
])

y_train = np.array([
    "A",
    "A",
    "A",
    "B",
    "B",
    "B"
])

In [6]:
def calculate_distances(X_train, x, metric="euclidean"):
    """
    Calculate the distance between x and every point in X_train.
    """

    distances = []

    for point in X_train:

        if metric == "euclidean":
            distance = euclidean_distance(point, x)

        elif metric == "manhattan":
            distance = manhattan_distance(point, x)

        else:
            raise ValueError(
                "metric must be 'euclidean' or 'manhattan'"
            )

        distances.append(distance)

    return np.array(distances)

In [7]:
x = np.array([2, 2])

distances = calculate_distances(
    X_train,
    x,
    metric="euclidean"
)

print(distances)

[1.41421356 1.         1.         4.24264069 5.         5.        ]


In [8]:
nearest_indices = np.argsort(distances)
print(nearest_indices)
print(distances[nearest_indices])

[1 2 0 3 4 5]
[1.         1.         1.41421356 4.24264069 5.         5.        ]


In [9]:
k = 3

nearest_indices = np.argsort(distances)
k_nearest_indices = nearest_indices[:k]

print(k_nearest_indices)

[1 2 0]


In [10]:
neighbor_labels = y_train[k_nearest_indices]

print(neighbor_labels)

['A' 'A' 'A']


In [11]:
def majority_vote(labels):
    """
    Return the most common label.
    """

    unique_labels, counts = np.unique(
        labels,
        return_counts=True
    )

    return unique_labels[np.argmax(counts)]

In [12]:
labels = np.array(["A", "B", "A", "A", "B"])

print(majority_vote(labels))

A


In [13]:
# KNN Classifier

class KNNClassifier:
    
    def __init__(self, k=3, metric="euclidean"):
        
        if k <= 0:
            raise ValueError("k must be greater than 0")
        
        if metric not in ["euclidean", "manhattan"]:
            raise ValueError(
                "metric must be 'euclidean' or 'manhattan'"
            )
        
        self.k = k
        self.metric = metric

    def fit(self, X, y):
        
        X = np.asarray(X)
        y = np.asarray(y)

        if len(X) != len(y):
            raise ValueError(
                "X and y must contain the same number of samples"
            )

        if self.k > len(X):
            raise ValueError(
                "k cannot be larger than the number of training samples"
            )

        self.X_train = X
        self.y_train = y

        return self

    def predict_one(self, x):
        
        distances = calculate_distances(
            self.X_train,
            x,
            metric=self.metric
        )

        nearest_indices = np.argsort(distances)[:self.k]

        neighbor_labels = self.y_train[nearest_indices]

        return majority_vote(neighbor_labels)

    def predict(self, X):
        
        X = np.asarray(X)

        predictions = [
            self.predict_one(x)
            for x in X
        ]

        return np.asarray(predictions)

In [14]:
knn = KNNClassifier(
    k=3,
    metric="euclidean"
)

knn.fit(X_train, y_train)

In [15]:
X_test = np.array([
    [2, 2]
])

prediction = knn.predict(X_test)

print(prediction)

['A']


In [16]:
X_test = np.array([
    [2, 2],
    [5, 5],
    [3, 3],
    [4, 4]
])

predictions = knn.predict(X_test)

print(predictions)

['A' 'B' 'A' 'B']


In [17]:
x = np.array([[3.5, 3.5]])

for k in [1, 3, 5]:
    
    model = KNNClassifier(
        k=k,
        metric="euclidean"
    )
    
    model.fit(X_train, y_train)
    
    prediction = model.predict(x)
    
    print(f"K={k}: {prediction[0]}")

K=1: B
K=3: A
K=5: B


In [18]:
'''
Small K
   ↓
Very local
   ↓
Sensitive to noise
   ↓
Higher variance

Large K
   ↓
Very broad neighborhood
   ↓
Local patterns ignored
   ↓
Higher bias

'''


'\nSmall K\n   ↓\nVery local\n   ↓\nSensitive to noise\n   ↓\nHigher variance\n\nLarge K\n   ↓\nVery broad neighborhood\n   ↓\nLocal patterns ignored\n   ↓\nHigher bias\n\n'

In [19]:
def accuracy_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return np.mean(y_true == y_pred)

In [20]:
# compare our implementation with Scikit-learn

from sklearn.neighbors import KNeighborsClassifier

sklearn_knn = KNeighborsClassifier(
    n_neighbors=3,
    metric="euclidean"
)

sklearn_knn.fit(
    X_train,
    y_train
)

sklearn_predictions = sklearn_knn.predict(X_test)

print(sklearn_predictions)

['A' 'B' 'A' 'B']


In [21]:
my_predictions = knn.predict(X_test)

print("Our implementation:")
print(my_predictions)

print("\nScikit-learn:")
print(sklearn_predictions)

Our implementation:
['A' 'B' 'A' 'B']

Scikit-learn:
['A' 'B' 'A' 'B']


In [22]:
'''
    KNN depends directly on distances, so feature scale affects its predictions.
'''

X = np.array([
    [20, 80000],
    [25, 60000],
    [30, 70000],
    [35, 40000],
    [40, 50000],
    [45, 30000]
])

y = np.array([
    "A",
    "A",
    "A",
    "B",
    "B",
    "B"
])

x_new = np.array([32, 55000])

distances = calculate_distances(
    X,
    x_new,
    metric="euclidean"
)

for i, distance in enumerate(distances):
    print(
        f"Point {i}: "
        f"{X[i]} → "
        f"distance = {distance:.2f}, "
        f"class = {y[i]}"
    )

Point 0: [   20 80000] → distance = 25000.00, class = A
Point 1: [   25 60000] → distance = 5000.00, class = A
Point 2: [   30 70000] → distance = 15000.00, class = A
Point 3: [   35 40000] → distance = 15000.00, class = B
Point 4: [   40 50000] → distance = 5000.01, class = B
Point 5: [   45 30000] → distance = 25000.00, class = B


In [23]:
def standardize(X):
    X = np.asarray(X, dtype=float)

    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)

    X_scaled = (X - mean) / std

    return X_scaled

X_scaled = standardize(X)

print(X_scaled)

[[-1.46385011  1.46385011]
 [-0.87831007  0.29277002]
 [-0.29277002  0.87831007]
 [ 0.29277002 -0.87831007]
 [ 0.87831007 -0.29277002]
 [ 1.46385011 -1.46385011]]


In [24]:
print("Means:")
print(np.mean(X_scaled, axis=0))
print("Standard deviations:")
print(np.std(X_scaled, axis=0))

Means:
[0.00000000e+00 3.70074342e-17]
Standard deviations:
[1. 1.]


In [27]:
x_new = np.array([32, 55000])

unscaled_distances = calculate_distances(
    X,
    x_new,
    metric="euclidean"
)

mean = np.mean(X, axis=0)
std = np.std(X, axis=0)

x_new_scaled = (x_new - mean) / std

scaled_distances = calculate_distances(
    X_scaled,
    x_new_scaled,
    metric="euclidean"
)

In [28]:
for i in range(len(X)):

    print(
        f"Point {i}: "
        f"Unscaled = {unscaled_distances[i]:.2f}, "
        f"Scaled = {scaled_distances[i]:.2f}, "
        f"Class = {y[i]}"
    )

Point 0: Unscaled = 25000.00, Scaled = 2.03, Class = A
Point 1: Unscaled = 5000.00, Scaled = 0.87, Class = A
Point 2: Unscaled = 15000.00, Scaled = 0.91, Class = A
Point 3: Unscaled = 15000.00, Scaled = 0.95, Class = B
Point 4: Unscaled = 5000.01, Scaled = 0.98, Class = B
Point 5: Unscaled = 25000.00, Scaled = 2.11, Class = B


In [ ]:
'''

We don't scale because:
"KNN requires values between 0 and 1."

That statement would be incorrect.
There are several possible scaling methods.
For example:
Standardization \[z=\frac{x-\mu}{\sigma}\]
does not force values into: \[[0,1]\]

Min-Max scaling
Instead: \[\boxed{x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}}\]
maps the training feature values to: \[[0,1]\]

'''

In [29]:
def minmax_scale(X):
    X = np.asarray(X, dtype=float)

    minimum = np.min(X, axis=0)
    maximum = np.max(X, axis=0)

    X_scaled = (
        (X - minimum)
        / (maximum - minimum)
    )

    return X_scaled

In [30]:
X_minmax = minmax_scale(X)

print(X_minmax)

[[0.  1. ]
 [0.2 0.6]
 [0.4 0.8]
 [0.6 0.2]
 [0.8 0.4]
 [1.  0. ]]


In [ ]:
'''

                    Dataset
                        │
             Train / Test Split
                        │
          ┌─────────────┼─────────────┐
          │             │             │
       No Scaling   StandardScaler  MinMaxScaler
          │             │             │
          └─────────────┼─────────────┘
                        ↓
                       KNN
                        │
                        ↓
                    Accuracy

'''